# The Fisher information matrix and the Hessian — experiments

Companion notebook to the post. Each experiment isolates one idea:

1. **Two faces of one matrix** — the variance of the score equals the negative
   expected Hessian of the log-likelihood. Verified for four models.
2. **Curvature at the MLE** — the log-likelihood's curvature *is* the observed
   information; the second-order Taylor expansion and its osculating circle of
   radius $1/I$. Sharp peak = high information = low variance.
3. **Inverse Fisher = covariance** — the Cramer-Rao bound and the asymptotic
   normality of the MLE, $\sqrt{n}(\hat\theta-\theta)\to\mathcal N(0,I_1^{-1})$.
4. **Fisher is the Hessian of the KL divergence** — the local quadratic that
   makes $I$ the natural metric on a statistical model.
5. **The matrix case** — the $2\times2$ information matrix of $\mathcal N(\mu,\sigma^2)$,
   its zero off-diagonal, and the asymptotic covariance ellipse of the MLE.
6. **A tensor, not a number** — reparameterization sends $I\mapsto J^\top I J$,
   which is exactly why the natural gradient is coordinate-free.
7. **The Hessian in machine learning** — for an NLL loss the Hessian splits into
   the Fisher (Gauss-Newton) term plus a residual term that averages to zero.

Pure `numpy` + `matplotlib`, styled to match the site.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Circle, Ellipse

# --- site theme -----------------------------------------------------------
BG, PANEL, LINE = "#0e1116", "#151a21", "#222a35"
FG, MUTED, ACCENT, ACCENT2 = "#d7dde6", "#8a94a3", "#4aa3ff", "#e0a458"
GREEN, RED, PURPLE = "#2a9d8f", "#d1495b", "#c792ea"

mpl.rcParams.update({
    "figure.facecolor": PANEL, "axes.facecolor": PANEL, "savefig.facecolor": PANEL,
    "axes.edgecolor": LINE, "axes.labelcolor": FG, "text.color": FG,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": LINE,
    "axes.grid": True, "grid.linewidth": 0.6, "grid.alpha": 0.35,
    "font.family": "monospace", "font.size": 11, "figure.dpi": 120,
    "axes.titlesize": 12, "legend.frameon": False, "legend.labelcolor": FG,
})

def style(ax):
    """Hide top/right spines, color the rest — the house look."""
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(LINE)
    return ax

ART = "artifacts"   # where figures are written
rng = np.random.default_rng(0)

## 1. Two faces of one matrix

The **score** is the gradient of the log-density, $s(\theta;x)=\nabla_\theta\log p_\theta(x)$.
Differentiating $\int p_\theta=1$ shows the score has mean zero, and that its two natural
second moments coincide:

$$ I(\theta)\;=\;\mathbb E\big[s\,s^\top\big]\;=\;-\,\mathbb E\big[\nabla^2_\theta\log p_\theta(x)\big]. $$

The left side is the variance of the score; the right side is the *expected curvature* of
the log-likelihood. They are equal because $\mathbb E[\nabla^2 p_\theta / p_\theta]=\nabla^2\!\int p_\theta=0$.
We check both estimators against the closed form for four standard models.

In [ ]:
# Each model: a sampler, an analytic score, an analytic 2nd derivative of log p,
# and the textbook Fisher information — all for a single observation.
def make_models():
    return {
        "Bernoulli(p=.3)": dict(
            theta=0.3,
            sample=lambda th, m, r: (r.random(m) < th).astype(float),
            score=lambda th, x: x/th - (1-x)/(1-th),
            d2=lambda th, x: -x/th**2 - (1-x)/(1-th)**2,
            fisher=lambda th: 1.0/(th*(1-th)),
        ),
        "Normal mean(mu=2,sd=1.5)": dict(
            theta=2.0, sd=1.5,
            sample=lambda th, m, r: r.normal(th, 1.5, m),
            score=lambda th, x: (x-th)/1.5**2,
            d2=lambda th, x: np.full_like(x, -1/1.5**2),
            fisher=lambda th: 1.0/1.5**2,
        ),
        "Poisson(lam=3)": dict(
            theta=3.0,
            sample=lambda th, m, r: r.poisson(th, m).astype(float),
            score=lambda th, x: x/th - 1.0,
            d2=lambda th, x: -x/th**2,
            fisher=lambda th: 1.0/th,
        ),
        "Exponential(lam=1.5)": dict(
            theta=1.5,
            sample=lambda th, m, r: r.exponential(1/th, m),
            score=lambda th, x: 1/th - x,
            d2=lambda th, x: np.full_like(x, -1/th**2),
            fisher=lambda th: 1.0/th**2,
        ),
    }

models = make_models()
N = 4_000_000
print(f"{'model':<26}{'closed form':>13}{'E[score^2]':>13}{'-E[hessian]':>13}")
print("-" * 65)
for name, M in models.items():
    th = M["theta"]
    x = M["sample"](th, N, rng)
    var_score = np.mean(M["score"](th, x)**2)        # E[s^2]  (score has mean 0)
    neg_hess = np.mean(-M["d2"](th, x))              # -E[d2 log p]
    closed = M["fisher"](th)
    print(f"{name:<26}{closed:>13.5f}{var_score:>13.5f}{neg_hess:>13.5f}")
print("\nBoth Monte-Carlo estimators land on the same closed form: "
      "variance of the score == negative expected Hessian.")

A convergence view of the identity: for the Poisson model, both Monte-Carlo estimators
of $I(\lambda)$ — the score-variance form and the negative-Hessian form — converge to the
same closed-form value $1/\lambda$.

In [ ]:
M = models["Poisson(lam=3)"]; th = M["theta"]
Ns = np.unique(np.round(np.logspace(1.3, 6, 40)).astype(int))
r2 = np.random.default_rng(7)
big = M["sample"](th, Ns.max(), r2)
sc2 = M["score"](th, big)**2
nh = -M["d2"](th, big)
est_score = np.array([sc2[:n].mean() for n in Ns])
est_hess = np.array([nh[:n].mean() for n in Ns])

fig, ax = plt.subplots(figsize=(7.2, 4.3))
ax.axhline(M["fisher"](th), color=MUTED, ls="--", lw=1.2, label="closed form  1/lambda")
ax.plot(Ns, est_score, color=ACCENT, lw=1.8, label="MC  E[score^2]")
ax.plot(Ns, est_hess, color=ACCENT2, lw=1.8, ls=":", label="MC  -E[hessian]")
ax.set_xscale("log"); ax.set_xlabel("sample size  N"); ax.set_ylabel("Fisher information  I(lambda)")
ax.set_ylim(0.28, 0.40); ax.set_title("two estimators, one quantity"); ax.legend(); style(ax)
plt.tight_layout(); plt.savefig(f"{ART}/fisher-two-forms.png", bbox_inches="tight"); plt.show()

## 2. Curvature at the MLE

Evaluate the negative Hessian of the log-likelihood **at the data**, not in expectation, and
you get the *observed information* $J_n(\hat\theta)=-\ell_n''(\hat\theta)$. Around the peak the
log-likelihood is, to second order,

$$ \ell_n(\theta)\;\approx\;\ell_n(\hat\theta)\;-\;\tfrac12\,(\theta-\hat\theta)^2\,J_n(\hat\theta). $$

Geometrically the curvature at the apex is $\kappa=J_n(\hat\theta)$, so the **osculating circle**
has radius $r=1/J_n$. Small information means a flat, wide peak — a large circle and a high-variance
estimate; large information means a sharp peak — a small circle and a precise estimate. This is the
picture, reproduced for an exponential model at two sample sizes.

In [ ]:
def loglik_exp(lam, xbar, n):
    return n * (np.log(lam) - lam * xbar)          # exponential, up to a constant

def panel(ax, n, lam_true=1.0, seed=1):
    r = np.random.default_rng(seed)
    x = r.exponential(1/lam_true, n)
    xbar = x.mean(); lam_hat = 1/xbar               # MLE
    J = n / lam_hat**2                              # observed information  -l''(lam_hat)
    R = 1.0 / J                                     # osculating radius
    g = np.linspace(lam_hat - 3.5*np.sqrt(1/J), lam_hat + 3.5*np.sqrt(1/J), 400)
    g = g[g > 1e-3]
    ll = loglik_exp(g, xbar, n); ll -= loglik_exp(lam_hat, xbar, n)   # peak at 0
    quad = -0.5 * J * (g - lam_hat)**2
    ax.plot(g, ll, color=ACCENT, lw=2.2, label="log-likelihood")
    ax.plot(g, quad, color=ACCENT2, lw=1.5, ls="--", label="2nd-order Taylor")
    ax.add_patch(Circle((lam_hat, -R), R, fill=False, ec=PURPLE, lw=1.4))
    ax.plot([lam_hat], [0], "o", color=FG, ms=5, zorder=5)
    ax.plot([lam_hat], [-R], "x", color=MUTED, ms=7)
    ax.set_aspect("equal")
    ax.set_xlim(lam_hat - 2.2*R, lam_hat + 2.2*R)
    ax.set_ylim(-2.4*R, 0.45*R)
    ax.set_title(f"n = {n}:   J = {J:.1f},  r = 1/J = {R:.3f}")
    ax.set_xlabel("lambda"); style(ax)
    return J

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.6))
panel(axL, n=6,  seed=3)
panel(axR, n=60, seed=3)
axL.set_ylabel("log-likelihood  (peak at 0)")
axL.text(0.05, 0.05, "Fisher small\nflat peak -> high variance", transform=axL.transAxes,
         color=MUTED, fontsize=9.5, va="bottom", ha="left", clip_on=False)
axR.text(0.05, 0.05, "Fisher large\nsharp peak -> low variance", transform=axR.transAxes,
         color=MUTED, fontsize=9.5, va="bottom", ha="left", clip_on=False)
axL.legend(loc="lower right", fontsize=9)
fig.suptitle("curvature at the MLE = observed information;  osculating radius = 1/J", y=1.0)
plt.tight_layout(); plt.savefig(f"{ART}/fisher-curvature.png", bbox_inches="tight"); plt.show()

# numeric check: the analytic observed information equals minus the numerical second derivative
r = np.random.default_rng(3); x = r.exponential(1.0, 60); xbar = x.mean(); lam_hat = 1/xbar
h = 1e-4
num_d2 = (loglik_exp(lam_hat+h, xbar, 60) - 2*loglik_exp(lam_hat, xbar, 60)
          + loglik_exp(lam_hat-h, xbar, 60)) / h**2
print(f"observed info  J = -l''(lam_hat):  analytic {60/lam_hat**2:.4f}   numerical {-num_d2:.4f}")

## 3. Inverse Fisher = covariance

Two consequences of the curvature picture. The **Cramer-Rao bound** says no unbiased estimator
can beat the inverse information, $\operatorname{Var}(\hat\theta)\ge 1/(n\,I_1)$, and the MLE is
**asymptotically efficient and normal**:

$$ \sqrt{n}\,(\hat\theta_n-\theta^\star)\ \xrightarrow{d}\ \mathcal N\!\big(0,\;I_1(\theta^\star)^{-1}\big). $$

For Poisson the MLE is the sample mean, $I_1=1/\lambda$, and the bound is met. We also watch the
*observed* information per sample, $J_n(\hat\lambda)/n$, converge to the *expected* information $I_1$.

In [ ]:
lam_true = 3.0; I1 = 1/lam_true
reps = 6000
ns = np.unique(np.round(np.logspace(0.7, 3.2, 18)).astype(int))
r3 = np.random.default_rng(11)
emp_var, mean_obs_over_n = [], []
for n in ns:
    data = r3.poisson(lam_true, size=(reps, n)).astype(float)
    lam_hat = data.mean(1)
    emp_var.append(lam_hat.var())
    mean_obs_over_n.append(np.mean((data.mean(1)) / lam_hat**2))   # J_n/n = lam_hat/lam_hat^2 = 1/lam_hat

emp_var = np.array(emp_var)
crb = 1.0 / (ns * I1)

# large-n sampling distribution of sqrt(n)(lam_hat - lam_true)
nbig = 400
lam_hat_big = r3.poisson(lam_true, size=(60000, nbig)).astype(float).mean(1)
z = np.sqrt(nbig) * (lam_hat_big - lam_true)
xg = np.linspace(z.min(), z.max(), 300)
dens = np.exp(-xg**2/(2*lam_true)) / np.sqrt(2*np.pi*lam_true)   # N(0, 1/I1 = lambda)

fig, (axA, axB) = plt.subplots(1, 2, figsize=(11, 4.3))
axA.plot(ns, crb, color=MUTED, ls="--", lw=1.6, label="Cramer-Rao  1/(n I1)")
axA.plot(ns, emp_var, "o-", color=ACCENT, lw=1.6, ms=4, label="empirical Var(lam_hat)")
axA.set_xscale("log"); axA.set_yscale("log")
axA.set_xlabel("sample size  n"); axA.set_ylabel("variance")
axA.set_title("MLE meets the Cramer-Rao floor"); axA.legend(); style(axA)

axB.hist(z, bins=70, density=True, color=ACCENT, alpha=0.45, edgecolor="none")
axB.plot(xg, dens, color=ACCENT2, lw=2.2, label="N(0, I1^-1)")
axB.set_xlabel("sqrt(n) (lam_hat - lambda)"); axB.set_ylabel("density")
axB.set_title(f"asymptotic normality (n={nbig})"); axB.legend(); style(axB)
plt.tight_layout(); plt.savefig(f"{ART}/fisher-crb.png", bbox_inches="tight"); plt.show()

print(f"expected info per sample I1 = 1/lambda = {I1:.4f}")
print(f"observed J_n/n -> I1:  at n={ns[-1]} mean J_n/n = {mean_obs_over_n[-1]:.4f}")
print(f"empirical Var * n at largest n: {emp_var[-1]*ns[-1]:.4f}   (CRB predicts 1/I1 = {1/I1:.4f})")

## 4. Fisher is the Hessian of the KL divergence

Why is $I$ the *natural* metric on a model and not just a variance? Expand the KL divergence to a
nearby distribution. The first-order term vanishes (KL is minimized at $\delta=0$) and the second-order
term is exactly the Fisher matrix:

$$ D_{\mathrm{KL}}\!\big(p_\theta\,\|\,p_{\theta+\delta}\big)\;=\;\tfrac12\,\delta^\top I(\theta)\,\delta\;+\;O(\|\delta\|^3). $$

Left: the Bernoulli KL against its quadratic, in one parameter. Right: for $\mathcal N(\mu,\sigma^2)$ the
exact KL contours and the Fisher ellipse $\tfrac12\delta^\top I\delta=c$ agree near the origin — the
information matrix is literally the local shape of KL.

In [ ]:
def kl_bern(p, q):
    return p*np.log(p/q) + (1-p)*np.log((1-p)/(1-q))

def kl_gauss(m0, v0, m1, v1):       # KL( N(m0,v0) || N(m1,v1) )
    return 0.5*(np.log(v1/v0) + (v0 + (m0-m1)**2)/v1 - 1.0)

# (left) 1D Bernoulli
p0 = 0.35
d = np.linspace(-0.22, 0.22, 400)
kl = kl_bern(p0, p0 + d)
quad = 0.5 * d**2 / (p0*(1-p0))     # 1/2 I(p) delta^2,  I(p)=1/(p(1-p))

# (right) 2D Gaussian (mu, sigma^2)
mu0, v0 = 0.0, 1.0
I2 = np.array([[1/v0, 0.0], [0.0, 1/(2*v0**2)]])     # Fisher of N(mu, sigma^2)
gm = np.linspace(-0.9, 0.9, 220); gv = np.linspace(-0.9, 0.9, 220)
DM, DV = np.meshgrid(gm, gv)
KLg = kl_gauss(mu0, v0, mu0 + DM, v0 + DV)
QUAD = 0.5*(I2[0,0]*DM**2 + 2*I2[0,1]*DM*DV + I2[1,1]*DV**2)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.4))
axL.plot(d, kl, color=ACCENT, lw=2.3, label="exact KL")
axL.plot(d, quad, color=ACCENT2, lw=1.6, ls="--", label="1/2 I(p) delta^2")
axL.axvline(0, color=MUTED, lw=0.8, alpha=0.5)
axL.set_xlabel("delta  (shift in p)"); axL.set_ylabel("KL  (nats)")
axL.set_title(f"Bernoulli, p={p0}"); axL.legend(); style(axL)

levels = [0.02, 0.08, 0.2, 0.4]
cs1 = axR.contour(DM, DV, KLg, levels=levels, colors=ACCENT, linewidths=1.8)
cs2 = axR.contour(DM, DV, QUAD, levels=levels, colors=ACCENT2, linewidths=1.2, linestyles="--")
axR.plot([0], [0], "o", color=FG, ms=4)
axR.set_aspect("equal"); axR.set_xlabel("delta mu"); axR.set_ylabel("delta sigma^2")
axR.set_title("Gaussian: KL (solid) vs Fisher quadratic (dashed)")
h1 = plt.Line2D([], [], color=ACCENT, lw=1.8); h2 = plt.Line2D([], [], color=ACCENT2, lw=1.2, ls="--")
axR.legend([h1, h2], ["exact KL", "1/2 delta^T I delta"], loc="upper right", fontsize=9)
style(axR)
plt.tight_layout(); plt.savefig(f"{ART}/fisher-kl-hessian.png", bbox_inches="tight"); plt.show()

for dd in (0.05, 0.1):
    print(f"Bernoulli delta={dd}:  KL={kl_bern(p0,p0+dd):.6f}   1/2 I d^2={0.5*dd**2/(p0*(1-p0)):.6f}")

## 5. The matrix case

With more than one parameter the information becomes a genuine matrix. For
$\mathcal N(\mu,\sigma^2)$ with both unknown,

$$ I(\mu,\sigma^2)=\begin{pmatrix} 1/\sigma^2 & 0\\[2pt] 0 & 1/(2\sigma^4)\end{pmatrix}. $$

The zero off-diagonal says $\mu$ and $\sigma^2$ are *orthogonal* parameters — their estimators are
asymptotically uncorrelated. We confirm the matrix by Monte Carlo (score outer product **and** negative
Hessian), then show its inverse predicts the spread of the MLE: $\operatorname{Cov}(\hat\mu,\hat\sigma^2)\approx I^{-1}/n$.

In [ ]:
mu_t, v_t = 1.0, 4.0     # sigma^2 = 4
def score_normal(mu, v, x):              # gradient wrt (mu, sigma^2)
    return np.stack([(x-mu)/v, -0.5/v + (x-mu)**2/(2*v**2)], axis=-1)

# Monte-Carlo Fisher via outer product of the score
n_mc = 5_000_000
xs = rng.normal(mu_t, np.sqrt(v_t), n_mc)
S = score_normal(mu_t, v_t, xs)                       # (n_mc, 2)
I_outer = (S[:, :, None] * S[:, None, :]).mean(0)

# Monte-Carlo Fisher via negative Hessian of log p (analytic 2nd derivatives)
d_mumu = -1/v_t
d_muv = -(xs - mu_t)/v_t**2
d_vv = 0.5/v_t**2 - (xs - mu_t)**2/v_t**3
H = np.empty((2, 2))
H[0, 0] = np.mean(-d_mumu); H[0, 1] = H[1, 0] = np.mean(-d_muv); H[1, 1] = np.mean(-d_vv)

I_closed = np.array([[1/v_t, 0.0], [0.0, 1/(2*v_t**2)]])
np.set_printoptions(precision=5, suppress=True)
print("closed-form Fisher:\n", I_closed)
print("E[score score^T] :\n", I_outer)
print("-E[Hessian]      :\n", H)

# asymptotic covariance of the MLE vs I^-1 / n
n = 200; reps = 40000
data = rng.normal(mu_t, np.sqrt(v_t), size=(reps, n))
mu_hat = data.mean(1)
v_hat = data.var(1)                                   # MLE uses 1/n
emp_cov = np.cov(np.stack([mu_hat, v_hat]))
pred_cov = np.linalg.inv(I_closed) / n
print("\nempirical Cov(mu_hat, v_hat):\n", emp_cov)
print("predicted  I^-1 / n        :\n", pred_cov)

# confidence ellipse from the predicted covariance over the empirical scatter
def ellipse(ax, cov, center, nstd2=5.991, **kw):      # chi^2_2(0.95)=5.991
    vals, vecs = np.linalg.eigh(cov)
    ang = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    w, h = 2*np.sqrt(vals*nstd2)
    ax.add_patch(Ellipse(center, w, h, angle=ang, fill=False, **kw))

fig, ax = plt.subplots(figsize=(7.0, 4.6))
idx = rng.choice(reps, 4000, replace=False)
ax.scatter(mu_hat[idx], v_hat[idx], s=5, color=ACCENT, alpha=0.25, edgecolor="none")
ellipse(ax, pred_cov, (mu_t, v_t), ec=ACCENT2, lw=2.2)
ax.plot([mu_t], [v_t], "x", color=FG, ms=9)
ax.set_xlabel("mu_hat"); ax.set_ylabel("sigma^2_hat  (= v_hat)")
ax.set_title(f"MLE scatter (n={n}) with 95% ellipse from I^-1/n"); style(ax)
plt.tight_layout(); plt.savefig(f"{ART}/fisher-matrix-ellipse.png", bbox_inches="tight"); plt.show()

## 6. A tensor, not a number

The information is not attached to "the parameter" but to the **distribution**: change coordinates
$\phi=\phi(\theta)$ and it transforms as a metric, $I_\phi = J^\top I_\theta\,J$ with $J=\partial\theta/\partial\phi$.
For Bernoulli, $I(p)=1/(p(1-p))$ explodes at the edges, while in the natural parameter
$\eta=\log\frac{p}{1-p}$ it is the tame $I(\eta)=p(1-p)$. The payoff: the **natural gradient**
$I^{-1}\nabla L$ produces the *same* update on the distribution in either coordinate system, whereas
the ordinary gradient does not.

In [ ]:
ps = np.linspace(0.02, 0.98, 400)
Ip = 1/(ps*(1-ps))                 # Fisher in p
Ieta = ps*(1-ps)                   # Fisher in eta = logit(p):  J=dp/deta=p(1-p), Ieta = J^2 Ip = p(1-p)

# tensor law check on a grid:  I_eta == (dp/deta)^2 I_p
J = ps*(1-ps)
print("max |I_eta - J^2 I_p| over grid:", np.max(np.abs(Ieta - J**2 * Ip)))

# natural-gradient invariance at one point, NLL with (a heads, b tails)
p = 0.30; a, b = 7, 3; alpha = 0.1
grad_p = -(a/p - b/(1-p))                       # dL/dp
grad_eta = (p*(1-p)) * grad_p                   # dL/deta = (dp/deta) dL/dp
# plain gradient steps -> induced change in p
dp_plain_p = -alpha*grad_p
dp_plain_eta = (p*(1-p)) * (-alpha*grad_eta)    # map eta-step back to p (first order)
# natural gradient steps -> induced change in p
dp_nat_p = -alpha * (p*(1-p)) * grad_p          # I_p^-1 grad_p, I_p^-1 = p(1-p)
deta_nat = -alpha * (1/(p*(1-p))) * grad_eta    # I_eta^-1 grad_eta
dp_nat_eta = (p*(1-p)) * deta_nat               # map back to p
print(f"\nplain   grad:  delta-p from p-coords = {dp_plain_p:+.6f}   from eta-coords = {dp_plain_eta:+.6f}  (differ)")
print(f"natural grad:  delta-p from p-coords = {dp_nat_p:+.6f}   from eta-coords = {dp_nat_eta:+.6f}  (identical)")

fig, ax = plt.subplots(figsize=(7.2, 4.3))
ax.plot(ps, Ip, color=ACCENT, lw=2.3, label="I(p) = 1/(p(1-p))")
ax.plot(ps, Ieta, color=ACCENT2, lw=2.3, label="I(eta) = p(1-p)")
ax.set_yscale("log"); ax.set_ylim(0.05, 60)
ax.set_xlabel("p"); ax.set_ylabel("Fisher information")
ax.set_title("same model, two coordinates: I = J^T I J"); ax.legend(); style(ax)
plt.tight_layout(); plt.savefig(f"{ART}/fisher-reparam.png", bbox_inches="tight"); plt.show()

## 7. The Hessian in machine learning

Train with a negative log-likelihood loss and the Fisher matrix is hiding in the Hessian. For
logistic regression the loss Hessian is $X^\top\!\operatorname{diag}\big(\sigma_i(1-\sigma_i)\big)X$,
which is *exactly* the (conditional) Fisher — the model is linear in $\theta$, so no residual term
appears and the Hessian is automatically positive semidefinite.

For a **nonlinear** model the Hessian splits in two,

$$ \nabla^2 L \;=\; \underbrace{\tfrac1{\sigma^2}\sum_i \nabla f_i\,\nabla f_i^\top}_{\text{Fisher / Gauss-Newton}} \;-\; \underbrace{\tfrac1{\sigma^2}\sum_i r_i\,\nabla^2 f_i}_{\text{residual term}}, $$

and the residual term has zero mean under the model, so as training drives the residuals down the
Hessian collapses onto the Fisher matrix.

In [ ]:
# --- (a) logistic regression: Hessian == conditional Fisher (machine precision)
rL = np.random.default_rng(5)
n, d = 800, 4
Xl = np.c_[np.ones(n), rL.normal(size=(n, d-1))]
w_true = rL.normal(size=d)
yl = (rL.random(n) < 1/(1+np.exp(-Xl @ w_true))).astype(float)

def sigmoid(z): return 1/(1+np.exp(-z))
w = np.zeros(d)
for _ in range(50):                                   # Newton / IRLS to the MLE
    pri = sigmoid(Xl @ w); Wd = pri*(1-pri)
    grad = Xl.T @ (pri - yl)
    Hess = Xl.T @ (Wd[:, None]*Xl)
    w -= np.linalg.solve(Hess + 1e-9*np.eye(d), grad)

pri = sigmoid(Xl @ w); Wd = pri*(1-pri)
H_logreg = Xl.T @ (Wd[:, None]*Xl)                    # Hessian of the NLL
F_cond = Xl.T @ (Wd[:, None]*Xl)                      # conditional Fisher  sum sigma(1-sigma) x x^T
print("logistic regression:  || Hessian - conditional Fisher || =",
      np.linalg.norm(H_logreg - F_cond))

# --- (b) nonlinear regression f(x;a,b)=a*exp(-b*x^2): Hessian -> Gauss-Newton as residuals shrink
rN = np.random.default_rng(2)
m = 200; xN = rN.uniform(-3, 3, m); a_t, b_t = 2.0, 0.6; sd = 0.2
yN = a_t*np.exp(-b_t*xN**2) + rN.normal(0, sd, m)
def f(th, x): return th[0]*np.exp(-th[1]*x**2)
def grad_f(th, x):                                    # d f / d(a,b)
    e = np.exp(-th[1]*x**2)
    return np.stack([e, -th[0]*x**2*e], axis=-1)
def loss(th): return 0.5*np.sum((yN - f(th, xN))**2)/sd**2

def num_hess(fn, th, h=1e-4):
    H = np.zeros((2, 2))
    for i in range(2):
        for j in range(2):
            tpp = th.copy(); tpp[i]+=h; tpp[j]+=h
            tpm = th.copy(); tpm[i]+=h; tpm[j]-=h
            tmp = th.copy(); tmp[i]-=h; tmp[j]+=h
            tmm = th.copy(); tmm[i]-=h; tmm[j]-=h
            H[i, j] = (fn(tpp)-fn(tpm)-fn(tmp)+fn(tmm))/(4*h*h)
    return H

th = np.array([0.5, 0.2])                             # poor start: large residuals
gaps, losses = [], []
for it in range(16):
    Jf = grad_f(th, xN)                               # (m,2)
    r = yN - f(th, xN)
    g = -(Jf * r[:, None]).sum(0)/sd**2
    GN = (Jf[:, :, None]*Jf[:, None, :]).sum(0)/sd**2   # Gauss-Newton == Fisher
    H_full = num_hess(loss, th)
    gaps.append(np.linalg.norm(H_full - GN)/np.linalg.norm(H_full))
    losses.append(loss(th))
    th = th - np.linalg.solve(GN + 1e-3*np.eye(2), g)   # GN / natural-gradient step

fig, ax = plt.subplots(figsize=(7.4, 4.3))
ax.plot(gaps, color=ACCENT, lw=2.2)
ax.set_yscale("log"); ax.set_xlabel("Gauss-Newton iteration")
ax.set_ylabel("|| H - Fisher || / || H ||")
ax.set_title("nonlinear regression: the residual term vanishes, H -> Fisher"); style(ax)
axr = ax.twinx()
axr.plot(losses, color=ACCENT2, lw=1.6, ls="--")
axr.set_yscale("log"); axr.set_ylabel("training loss", color=ACCENT2)
axr.tick_params(axis="y", colors=ACCENT2); axr.grid(False)
axr.spines["top"].set_visible(False)
plt.tight_layout(); plt.savefig(f"{ART}/fisher-gauss-newton.png", bbox_inches="tight"); plt.show()
print(f"\nnonlinear fit converged to a={th[0]:.3f}, b={th[1]:.3f}  (true {a_t}, {b_t})")
print(f"||H - Fisher||/||H||:  start {gaps[0]:.3f}  ->  end {gaps[-1]:.2e}")

## Takeaways

- The Fisher matrix has **two equal faces**: the variance of the score and the negative expected
  Hessian of the log-likelihood.
- At the MLE, **curvature is information**: the observed information is the negative Hessian of the
  log-likelihood, and its inverse is the asymptotic variance — a flat peak means an imprecise estimate.
- $I^{-1}$ is the **Cramer-Rao floor** and the **asymptotic covariance** of the MLE.
- $I$ is the **Hessian of the KL divergence**, which is what makes it the natural Riemannian metric on
  a statistical model.
- It is a **tensor**: $I_\phi=J^\top I_\theta J$, so the natural gradient $I^{-1}\nabla L$ is coordinate-free.
- In machine learning the **NLL Hessian equals the Fisher / Gauss-Newton matrix** plus a residual term
  that averages to zero, which is why the Fisher matrix is the curvature object behind natural-gradient
  and second-order optimizers.